# Notebook 1: Data Preprocessing & Descriptive Statistics**PhD Research** — Probabilistic and Statistical Analysis of the Menstrual Cycle from a Halachic Perspective  **Author:** Dvir Ross, Shenkar College  This notebook covers:1. Raw data loading and quality checks2. Data cleaning and filtering pipeline3. Comprehensive descriptive statistics4. Per-client cycle statistics5. Visualizations of cycle length distribution

In [ ]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport matplotlib.ticker as mtickerimport scipy.stats as statsimport warningswarnings.filterwarnings('ignore')# ── Plotting style ──────────────────────────────────────────────────────────plt.rcParams.update({    'figure.dpi': 120,    'font.size': 12,    'axes.titlesize': 14,    'axes.labelsize': 12,    'legend.fontsize': 11,    'axes.spines.top': False,    'axes.spines.right': False,})

## 1. Load Raw Data

In [ ]:
raw = pd.read_csv('datasets/RawData.csv')print(f"Raw dataset: {raw.shape[0]:,} rows × {raw.shape[1]} columns")print(f"Unique clients: {raw['ClientID'].nunique()}")print(f"Cycle number range: {raw['CycleNumber'].min()} – {raw['CycleNumber'].max()}")raw[['ClientID','CycleNumber','LengthofCycle','Age','BMI']].head(10)

## 2. Data Cleaning Pipeline**Exclusion criteria:**- Clients with duplicate (ClientID, CycleNumber) pairs (data integrity)- Known data anomaly: client `nfp8107` rows 45–53 (manual correction)- Clients with fewer than 5 cycles (insufficient longitudinal data)

In [ ]:
# ── Step 1: Identify and report duplicates ───────────────────────────────dup_mask = raw.duplicated(subset=['ClientID','CycleNumber'], keep=False)dup_clients = raw.loc[dup_mask, 'ClientID'].unique()print(f"Clients with duplicate cycle numbers: {len(dup_clients)}")print(f"Duplicate rows: {dup_mask.sum()}")# ── Step 2: Remove duplicate rows (keep first occurrence) ────────────────df = raw.drop_duplicates(subset=['ClientID','CycleNumber'], keep='first').copy()# ── Step 3: Manual correction for nfp8107 ────────────────────────────────if 'nfp8107' in df['ClientID'].values:    client_mask = df['ClientID'] == 'nfp8107'    client_data = df[client_mask]    drop_idx = client_data.index[44:53]  # rows 45–53 (0-indexed: 44–52)    df = df.drop(index=drop_idx)    print(f"Removed {len(drop_idx)} anomalous rows for nfp8107")# ── Step 4: Filter clients with >= 5 cycles ──────────────────────────────cycle_counts = df.groupby('ClientID')['CycleNumber'].count()valid_clients = cycle_counts[cycle_counts >= 5].indexdf = df[df['ClientID'].isin(valid_clients)].copy()df.reset_index(drop=True, inplace=True)print(f"\nFiltered dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")print(f"Valid clients: {df['ClientID'].nunique()}")print(f"Cycles per client — mean: {cycle_counts[valid_clients].mean():.1f}, "      f"median: {cycle_counts[valid_clients].median():.0f}, "      f"max: {cycle_counts[valid_clients].max()}")

## 3. Halachic Cycle Length ConventionIn Jewish law (Halacha), the day menstruation begins is counted as day 1 of the **new** cycle.Therefore, the Halachic interval (Haflaga) equals the calendar cycle length + 1.

In [ ]:
# Apply Halachic +1 correctiondf['LengthofCycle_Halachic'] = df['LengthofCycle'] + 1print("Cycle length statistics (Halachic convention):")print(df['LengthofCycle_Halachic'].describe().round(3))

## 4. Descriptive Statistics

In [ ]:
print("=" * 55)print("  DESCRIPTIVE STATISTICS — CYCLE LENGTH (HALACHIC)")print("=" * 55)L = df['LengthofCycle_Halachic']desc = {    'N (cycles)': len(L),    'N (clients)': df['ClientID'].nunique(),    'Mean': L.mean(),    'SD': L.std(ddof=1),    'Median': L.median(),    'IQR (Q1–Q3)': f"{L.quantile(0.25):.1f} – {L.quantile(0.75):.1f}",    'Min': L.min(),    'Max': L.max(),    'Skewness': L.skew(),    'Kurtosis (excess)': L.kurt(),}for k, v in desc.items():    if isinstance(v, float):        print(f"  {k:<22}: {v:.4f}")    else:        print(f"  {k:<22}: {v}")

In [ ]:
# ── Per-client statistics ─────────────────────────────────────────────────per_client = df.groupby('ClientID')['LengthofCycle_Halachic'].agg(    n_cycles='count',    mean='mean',    std='std',    cv=lambda x: x.std(ddof=1) / x.mean() * 100).round(3)print("Per-client cycle statistics (first 10 clients):")print(per_client.head(10).to_string())print(f"\nOverall within-client mean CV: {per_client['cv'].mean():.2f}%")

In [ ]:
# ── Shapiro-Wilk normality test on cycle lengths ─────────────────────────stat, p = stats.shapiro(df['LengthofCycle_Halachic'])print(f"Shapiro-Wilk test for cycle length normality:")print(f"  W = {stat:.6f},  p = {p:.2e}")print(f"  Conclusion: {'Non-normal' if p < 0.05 else 'Normal'} distribution (α = 0.05)")

## 5. Visualizations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))# ── (a) Histogram of cycle lengths ───────────────────────────────────────ax = axes[0]ax.hist(df['LengthofCycle_Halachic'], bins=30, edgecolor='white',        color='steelblue', alpha=0.85)ax.axvline(L.mean(), color='crimson', ls='--', lw=1.5, label=f'Mean = {L.mean():.1f}')ax.axvline(L.median(), color='darkorange', ls=':', lw=1.5, label=f'Median = {L.median():.0f}')ax.set_xlabel('Cycle Length (Halachic, days)')ax.set_ylabel('Frequency')ax.set_title('(a) Distribution of Cycle Lengths')ax.legend()# ── (b) Cycles per client ────────────────────────────────────────────────ax = axes[1]cycles_per_client = df.groupby('ClientID').size()ax.hist(cycles_per_client, bins=20, edgecolor='white', color='teal', alpha=0.85)ax.set_xlabel('Number of Cycles per Client')ax.set_ylabel('Number of Clients')ax.set_title('(b) Cycles per Client')ax.axvline(cycles_per_client.mean(), color='crimson', ls='--', lw=1.5,           label=f'Mean = {cycles_per_client.mean():.1f}')ax.legend()# ── (c) Per-client mean cycle length ─────────────────────────────────────ax = axes[2]ax.hist(per_client['mean'], bins=20, edgecolor='white', color='mediumpurple', alpha=0.85)ax.set_xlabel('Per-Client Mean Cycle Length (days)')ax.set_ylabel('Number of Clients')ax.set_title('(c) Per-Client Mean Cycle Length')plt.tight_layout()plt.savefig('figures/fig1_descriptive_statistics.png', bbox_inches='tight')plt.show()print("Figure saved to figures/fig1_descriptive_statistics.png")

In [ ]:
# ── Frequency table: cycle length distribution ────────────────────────────freq = df['LengthofCycle_Halachic'].value_counts().sort_index()freq_pct = (freq / len(df) * 100).round(2)freq_table = pd.DataFrame({'Count': freq, 'Percentage (%)': freq_pct})freq_table.index.name = 'Cycle Length (days)'print("Cycle length frequency table:")print(freq_table.to_string())

## 6. Export Filtered Dataset

In [ ]:
# Save filtered data (without Halachic column — that is computed on load)df_export = df.drop(columns=['LengthofCycle_Halachic'])df_export.to_csv('datasets/FilteredData.csv', index=False)print(f"Saved FilteredData.csv: {df_export.shape[0]:,} rows × {df_export.shape[1]} columns")